# Push-T × JEPA + **SIGReg officiel** (LeJEPA) — face à DINO-WM (SR 0.90)

World model **JEPA conjoint** (encodeur ViT + dynamique conditionnée par l'action), anti-collapse = **SIGReg du package `lejepa`** (aucun slot, aucune reconstruction, aucun décodeur). Planification **purement latente** (protocole DINO-WM).

Workflow : **1** (code+deps) → **2** (Drive) → **3** collecte (une fois) → **4** entraînement → **5** reprise → **6** éval MPC/oracle/aléa (coût latent) → **7** éval coût latent-sans-agent.

In [ ]:
# 1) Code + dépendances (gym-pusht + lejepa officiel)
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install --quiet gym-pusht 'pymunk<7'
!pip install -q lejepa || pip install -q git+https://github.com/rbalestr-lab/lejepa
import torch, lejepa
t = lejepa.univariate.EppsPulley(t_max=3, n_points=17)
f = lejepa.multivariate.SlicingUnivariateTest(univariate_test=t, num_slices=1024)
z = torch.randn(128,128, requires_grad=True); f(z).backward()
print('OK — lejepa grad ok =', z.grad is not None)

In [ ]:
# 2) Drive (données + checkpoints persistants)
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('pas de Drive:', e)

In [ ]:
# 3) COLLECTE (une fois — sauter si pusht_data.npz est déjà sur Drive)
#!python -u pusht_data.py --n 10000 --T 6

In [ ]:
# 4) ENTRAÎNEMENT depuis zéro (JEPA conjoint + SIGReg rw=1.0)
!python -u pusht_jepa.py --steps 30000 --bs 32 --rw 1.0

In [ ]:
# 5) REPRISE du checkpoint (architecture auto-lue)
#!python -u pusht_jepa.py --resume 1 --steps 20000

In [ ]:
# 6) ÉVAL — PLANIF latente pleine (fidèle DINO-WM) : MPC vs ORACLE vs aléatoire
#    but par image, ≤100 pas, succès = coverage officiel (>0.95)
!python -u pusht_jepa.py --load 1 --tasks 10 --cost latent

In [ ]:
# 7) ÉVAL — variante coût 'sans agent' (tokens bleus du but exclus)
!python -u pusht_jepa.py --load 1 --tasks 10 --cost latent_noagent

---
## Pipeline **2 ÉTAPES** (encodeur gelé + dynamique) — pusht_jepa2.py

Le JEPA conjoint (au-dessus) échoue : SIGReg gaussianise mais rend l'encodeur anti-lisse (copy 0.60 > moyenne 0.42). Parade (vision LeCun/DINO-WM) : **geler** une tête de perception et imaginer dedans. **A** encodeur I-JEPA+SIGReg → gelé, **B** dynamique sur latents gelés, **C** planif.

In [ ]:
# A) ÉTAPE 1 — encodeur I-JEPA masqué + SIGReg (puis GELÉ) — sauve pusht_enc.pt
!python -u pusht_jepa2.py --stage enc --enc_steps 30000

In [ ]:
# B) ÉTAPE 2 — dynamique sur latents GELÉS — sauve pusht_dyn.pt
#    (affiche d'abord la SMOOTHNESS de l'encodeur : copy<<moyenne = bon signe)
!python -u pusht_jepa2.py --stage dyn --dyn_steps 20000

In [ ]:
# C) ÉVAL — planif CEM latente (encodeur gelé + dynamique) : MPC vs ORACLE vs aléa
!python -u pusht_jepa2.py --stage eval --tasks 10 --cost latent

In [ ]:
# C') diagnostic rapide de l'encodeur gelé seul (copy vs moyenne vs cos), sans planifier
#!python -u pusht_jepa2.py --stage dyn --dyn_steps 0

### FRAMESKIP — horizon utile (le bloc bouge vraiment)

Probe : sur 4 pas denses le bloc bouge 1-14px → coût plat → MPC=random. Parade standard (DINO-WM) : action **maintenue K pas** → chaque pas de planif couvre K pas d'env → le T se déplace franchement. **L'encodeur gelé est réutilisé tel quel** ; seule la dynamique se réentraîne.

In [ ]:
# FS-1) données frameskip=5 (actions maintenues 5 pas) -> pusht_data_fs5.npz
!python -u pusht_data.py --n 8000 --T 6 --frameskip 5

In [ ]:
# FS-2) dynamique sur données frameskip (encodeur pusht_enc.pt GELÉ réutilisé)
!python -u pusht_jepa2.py --stage dyn --dyn_steps 20000 \
    --data /content/drive/MyDrive/pusht_data_fs5.npz --dyn_ckpt /content/drive/MyDrive/pusht_dyn_fs5.pt

In [ ]:
# FS-3) PROBE alignement coût-vrai en frameskip (le bloc doit enfin varier : σ dist grande)
!python -u pusht_jepa2.py --stage probe --frameskip 5 --cost latent_noagent \
    --dyn_ckpt /content/drive/MyDrive/pusht_dyn_fs5.pt

In [ ]:
# FS-4) ÉVAL frameskip : MPC vs random (40 pas planif x5 = 200 pas env)
!python -u pusht_jepa2.py --stage eval --tasks 10 --frameskip 5 --max_steps 40 \
    --policies mpc,random --cost latent_noagent --dyn_ckpt /content/drive/MyDrive/pusht_dyn_fs5.pt
# puis comparer avec --cost latent (option A) si besoin

---
## Encodeur MUSCLÉ — données mélangées (Push-T + images + vidéos) + plus gros/long

Sonde pose : l'encodeur 60k-frames encode la position à **85px** / angle **79°** (≈ hasard) → sous-entraîné/étroit. Recette DINO-WM : encodeur gelé issu d'un pré-entraînement **large et divers**. On mélange Push-T + STL-10 (images naturelles) + UCF101 (vidéos), d 128→256, 30k→100k pas. Nouveau checkpoint **pusht_enc_big.pt** (l'ancien est gardé pour comparaison).

In [ ]:
# BIG-1) encodeur musclé sur données MÉLANGÉES (télécharge STL-10 ~2.6Go + UCF101-subset)
!python -u pusht_jepa2.py --stage enc --mix pusht,stl,video \
    --d 256 --nl 6 --enc_steps 100000 --lr 2e-4 --clip 1.0 --enc_ckpt /content/drive/MyDrive/pusht_enc_big.pt

In [ ]:
# BIG-2) RE-SONDE pose sur l'encodeur musclé : la position se décode-t-elle enfin ?
!python -u pusht_jepa2.py --stage pose_probe --data /content/drive/MyDrive/pusht_data.npz \
    --enc_ckpt /content/drive/MyDrive/pusht_enc_big.pt

In [ ]:
# BIG-3) si la sonde est bonne : dynamique frameskip sur l'encodeur musclé
!python -u pusht_jepa2.py --stage dyn --dyn_steps 20000 \
    --data /content/drive/MyDrive/pusht_data_fs5.npz --enc_ckpt /content/drive/MyDrive/pusht_enc_big.pt --dyn_ckpt /content/drive/MyDrive/pusht_dyn_big.pt

In [ ]:
# BIG-4) probe alignement coût + éval MPC sur l'encodeur musclé
!python -u pusht_jepa2.py --stage probe --frameskip 5 --cost latent_noagent \
    --enc_ckpt /content/drive/MyDrive/pusht_enc_big.pt --dyn_ckpt /content/drive/MyDrive/pusht_dyn_big.pt
!python -u pusht_jepa2.py --stage eval --tasks 10 --frameskip 5 --max_steps 40 --policies mpc,random \
    --cost latent_noagent --enc_ckpt /content/drive/MyDrive/pusht_enc_big.pt --dyn_ckpt /content/drive/MyDrive/pusht_dyn_big.pt

---
## ABLATION V-JEPA 2 gelé (Meta) — notre encodeur est-il LE problème ?

On remplace notre encodeur maison par le **vrai V-JEPA 2** (ViT-L, gelé, 1M h vidéo) et on refait **la même sonde pose**. OUI ça décode → c'était notre « Chose 1 » ; NON → le problème est ailleurs. (= la séparation V-JEPA 2 → V-JEPA 2-AC de Meta.)

In [ ]:
# VJ2) sonde pose sur features V-JEPA 2 gelé (télécharge ViT-L ~1.2Go la 1re fois)
!pip install -q -U transformers
!python -u pusht_vjepa2.py --stage pose_probe --data /content/drive/MyDrive/pusht_data.npz

In [ ]:
# VJ2-cache) précalcule les features V-JEPA 2 des données frameskip -> memmap fp16 sur Drive
#   (~10-15 min, ViT-L ; ~9.4Go pour 3000 séq. Si Drive juste : ajoute --n_cache 2000)
!python -u pusht_vjepa2.py --stage cache --data /content/drive/MyDrive/pusht_data_fs5.npz

### Planificateur V-JEPA 2-AC (dynamique + readout-critic + éval)

Sur les features V-JEPA 2 en cache : **dyn** (dynamique action), **readout** (pose = coût/critic, validé 24px/10°), **eval** (CEM latent → imaginer → lire la pose → coût pose-vers-but → coverage vs DINO-WM 0.90).

In [ ]:
# VJ2-dyn) dynamique sur features V-JEPA 2 gelées (769 tokens x d1024 -> batch réduit)
#   RAM ~9.4Go (runtime high-RAM) ; si OOM GPU baisse encore --bs (8, 4)
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u pusht_vjepa2.py --stage dyn --dyn_steps 20000 --bs 16

In [ ]:
# VJ2-readout) readout de pose (critic) sur le cache — doit retrouver ~24px/10°
!python -u pusht_vjepa2.py --stage readout --readout_steps 6000

In [ ]:
# VJ2-eval) PLANIFICATION MPC (V-JEPA 2) vs random — coverage vs DINO-WM 0.90
#   si OOM GPU pendant le CEM, baisse --plan_pop (96, 48)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u pusht_vjepa2.py --stage eval --tasks 10 --policies mpc,random